### Задача 1. Расчёт DAU
``` sql
select 
    log_date,
    count(distinct user_id) as DAU
from analytics_events
join cities using(city_id)
where log_date >= '2021-05-01' and log_date <= '2021-06-30'
    and city_name = 'Саранск'
    and event = 'order'
group by log_date
order by log_date
limit 10
```
--- 

### Задача 2. Расчёт Conversion Rate
``` sql
select
    log_date,
    round((count(distinct user_id) filter (where event = 'order')) / count(distinct user_id)::numeric, 2) as CR
from analytics_events
join cities using(city_id)
where city_name = 'Саранск' and
    log_date >= '2021-05-01' and log_date <= '2021-06-30'
group by log_date
order by log_date
limit 10
```
---

### Задача 3. Расчёт среднего чека
``` sql
-- Рассчитываем величину комиссии с каждого заказа, отбираем заказы по дате и городу
WITH orders AS
    (SELECT *,
            revenue * commission AS commission_revenue
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE revenue IS NOT NULL
         AND log_date BETWEEN '2021-05-01' AND '2021-06-30'
         AND city_name = 'Саранск')

-- Напишите ваш код ниже

select 
    date_trunc('month', log_date)::date as "Месяц",
    count(distinct order_id) as "Количество заказов",
    round(sum(commission_revenue)::numeric, 2) as "Сумма комиссии",
    round((sum(commission_revenue) / count(distinct order_id))::numeric, 2) as "Средний чек"
from orders
group by date_trunc('month', log_date)::date
order by "Месяц"
```
---

### Задача 4. Расчёт LTV ресторанов
``` sql
-- Рассчитываем величину комиссии с каждого заказа, отбираем заказы по дате и городу
WITH orders AS
    (SELECT analytics_events.rest_id,
            analytics_events.city_id,
            revenue * commission AS commission_revenue
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE revenue IS NOT NULL
         AND log_date BETWEEN '2021-05-01' AND '2021-06-30'
         AND city_name = 'Саранск')

-- Напишите ваш код ниже

select
    o.rest_id,
    chain as "Название сети",
    type as "Тип кухни",
    round(sum(o.commission_revenue)::numeric, 2) as LTV
from orders o
join partners p on o.rest_id = p.rest_id and o.city_id = p.city_id
group by 1, 2, 3
order by LTV desc
limit 3
```
---

### Задача 5. Расчёт LTV ресторанов — самые популярные блюда
```sql
-- Рассчитываем величину комиссии с каждого заказа, отбираем заказы по дате и городу
WITH orders AS
    (SELECT analytics_events.rest_id,
            analytics_events.city_id,
            analytics_events.object_id,
            revenue * commission AS commission_revenue
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE revenue IS NOT NULL
         AND log_date BETWEEN '2021-05-01' AND '2021-06-30'
         AND city_name = 'Саранск'), 

-- Рассчитываем два ресторана с наибольшим LTV 
top_ltv_restaurants AS
    (SELECT orders.rest_id,
            chain,
            type,
            ROUND(SUM(commission_revenue)::numeric, 2) AS LTV
     FROM orders
     JOIN partners ON orders.rest_id = partners.rest_id AND orders.city_id = partners.city_id
     GROUP BY 1, 2, 3
     ORDER BY LTV DESC
     LIMIT 2)

-- Напишите ваш код ниже

select 
    chain as "Название сети",
    name as "Название блюда",
    spicy,
    fish,
    meat,
    round(sum(commission_revenue)::numeric, 2) as LTV
from orders o
join top_ltv_restaurants using(rest_id)
join dishes using(object_id)
group by 1, 2, 3, 4, 5
order by LTV desc
limit 5
```
---

### Задача 6. Расчёт Retention Rate
```sql
-- Рассчитываем новых пользователей по дате первого посещения продукта
WITH new_users AS
    (SELECT DISTINCT first_date,
                     user_id
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE first_date BETWEEN '2021-05-01' AND '2021-06-24'
         AND city_name = 'Саранск'),

-- Рассчитываем активных пользователей по дате события
active_users AS
    (SELECT DISTINCT log_date,
                     user_id
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE log_date BETWEEN '2021-05-01' AND '2021-06-30'
         AND city_name = 'Саранск'),

-- Напишите ваш код ниже
daily_retention AS (
SELECT n.user_id,
first_date,
log_date::date - first_date::date AS day_since_install
FROM new_users n
JOIN active_users a
on n.user_id = a.user_id
WHERE log_date >= first_date)


select 
    day_since_install,
    count(distinct user_id) as retained_users,
    ROUND(COUNT(DISTINCT user_id) * 1.0 / (SELECT COUNT(DISTINCT user_id) FROM new_users), 2) AS retention_rate
from daily_retention
WHERE day_since_install < 8
group by day_since_install
ORDER by day_since_install
```
---

### Задача 7. Сравнение Retention Rate по месяцам
```sql
-- Рассчитываем новых пользователей по дате первого посещения продукта
WITH new_users AS
    (SELECT DISTINCT first_date,
                     user_id
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE first_date BETWEEN '2021-05-01' AND '2021-06-24'
         AND city_name = 'Саранск'),

-- Рассчитываем активных пользователей по дате события
active_users AS
    (SELECT DISTINCT log_date,
                     user_id
     FROM analytics_events
     JOIN cities ON analytics_events.city_id = cities.city_id
     WHERE log_date BETWEEN '2021-05-01' AND '2021-06-30'
         AND city_name = 'Саранск'),

-- Соединяем таблицы с новыми и активными пользователями
daily_retention AS
    (SELECT new_users.user_id,
            first_date,
            log_date::date - first_date::date AS day_since_install
     FROM new_users
     JOIN active_users ON new_users.user_id = active_users.user_id
     AND log_date >= first_date)

-- Напишите ваш код ниже
select
    cast(date_trunc('month', first_date) as date) as "Месяц",
    day_since_install,
    count(distinct user_id) as retained_users,
    round(count(distinct user_id) * 1.0 / max(count(distinct user_id)) over(partition by cast(date_trunc('month', first_date) as date) order by day_since_install), 2) as retention_rate
from daily_retention
where day_since_install < 8
group by "Месяц", day_since_install
order by "Месяц", day_since_install
```
---

## Дашборд по результатам исследования

[Datalens](https://datalens.ru/u7xto6n0kqkuf?_theme=dark "Перейти")